## This notebook is just an experiment to do the bit-packing for Binary quantisation

In [ ]:
import numpy as np
import os
import tensorflow as tf
import struct
import json
import time

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/tinyml-quant-security'

# recreate the custom layers needed to load the model
@tf.custom_gradient
def binarize_ste(w):
    w_binarized = tf.sign(w)
    w_binarized = tf.where(
        tf.equal(w_binarized, 0),
        tf.ones_like(w_binarized),
        w_binarized
    )
    def grad(dy):
        mask = tf.cast(tf.abs(w) <= 1.0, dy.dtype)
        return dy * mask
    return w_binarized, grad

class BinaryConv2D(tf.keras.layers.Conv2D):
    def call(self, inputs):
        q_kernel = binarize_ste(self.kernel)
        outputs = tf.nn.conv2d(
            inputs, q_kernel,
            strides=[1, *self.strides, 1],
            padding=self.padding.upper()
        )
        if self.use_bias:
            outputs = tf.nn.bias_add(outputs, self.bias)
        if self.activation is not None:
            outputs = self.activation(outputs)
        return outputs

class BinaryDense(tf.keras.layers.Dense):
    def call(self, inputs):
        q_kernel = binarize_ste(self.kernel)
        outputs = tf.matmul(inputs, q_kernel)
        if self.use_bias:
            outputs = tf.nn.bias_add(outputs, self.bias)
        if self.activation is not None:
            outputs = self.activation(outputs)
        return outputs

# try loading .keras file with custom objects
custom_objects = {
    'BinaryConv2D': BinaryConv2D,
    'BinaryDense': BinaryDense,
    'binarize_ste': binarize_ste
}

try:
    # try .keras format first
    binary_model = tf.keras.models.load_model(
        f'{PROJECT_DIR}/models/binary_model.keras',
        custom_objects=custom_objects
    )
    print("Loaded binary_model.keras successfully")

except Exception as e:
    print(f"Could not load .keras: {e}")
    print("Trying TFSMLayer approach...")

    # fallback: load as inference-only layer using TFSMLayer
    binary_model = tf.keras.layers.TFSMLayer(
        f'{PROJECT_DIR}/models/binary_savedmodel',
        call_endpoint='serving_default'
    )
    print("Loaded via TFSMLayer")

# check what model files exist
print("\nModel files available:")
models_dir = f'{PROJECT_DIR}/models'
for f in os.listdir(models_dir):
    path = os.path.join(models_dir, f)
    if os.path.isfile(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"  {f}: {size_kb:.1f} KB")

In [ ]:
# recreate custom layers needed to load binary_model.keras
@tf.custom_gradient
def binarize_ste(w):
    w_binarized = tf.sign(w)
    w_binarized = tf.where(
        tf.equal(w_binarized, 0),
        tf.ones_like(w_binarized),
        w_binarized
    )
    def grad(dy):
        mask = tf.cast(tf.abs(w) <= 1.0, dy.dtype)
        return dy * mask
    return w_binarized, grad

class BinaryConv2D(tf.keras.layers.Conv2D):
    def call(self, inputs):
        q_kernel = binarize_ste(self.kernel)
        outputs = tf.nn.conv2d(
            inputs, q_kernel,
            strides=[1, *self.strides, 1],
            padding=self.padding.upper()
        )
        if self.use_bias:
            outputs = tf.nn.bias_add(outputs, self.bias)
        if self.activation is not None:
            outputs = self.activation(outputs)
        return outputs

class BinaryDense(tf.keras.layers.Dense):
    def call(self, inputs):
        q_kernel = binarize_ste(self.kernel)
        outputs = tf.matmul(inputs, q_kernel)
        if self.use_bias:
            outputs = tf.nn.bias_add(outputs, self.bias)
        if self.activation is not None:
            outputs = self.activation(outputs)
        return outputs

# load the binary model
custom_objects = {
    'BinaryConv2D': BinaryConv2D,
    'BinaryDense': BinaryDense,
    'binarize_ste': binarize_ste
}

binary_model = tf.keras.models.load_model(
    f'{PROJECT_DIR}/models/binary_model.keras',
    custom_objects=custom_objects
)
print("Loaded binary_model.keras successfully")

# check weights are ±1 in the binary layers
print("\nVerifying weights in binary layers:")
binary_layer_names = ['conv2', 'conv3', 'dense1']

for layer_name in binary_layer_names:
    try:
        layer = binary_model.get_layer(layer_name)
        weights = layer.get_weights()[0]

        # apply sign() to get the actual +-1 values
        binarized = np.sign(weights)
        binarized[binarized == 0] = 1.0

        unique_vals = np.unique(binarized)
        print(f"\n  {layer_name}:")
        print(f"    Shape:         {weights.shape}")
        print(f"    Unique vals:   {unique_vals}")
        print(f"    Original size: {weights.nbytes / 1024:.1f} KB")

        # calculate what bit-packing would give
        total_weights = np.prod(weights.shape)
        packed_bits_kb = total_weights / 8 / 1024  # 1 bit per weight
        print(f"    Bit-packed:    {packed_bits_kb:.1f} KB")
        print(f"    Compression:   {weights.nbytes / (packed_bits_kb * 1024):.0f}x")

    except Exception as e:
        print(f"  {layer_name}: {e}")

In [ ]:
# Here extract and pack all weights

def pack_binary_weights(float_weights):
    """
    Pack ±1 float32 weights into bit-packed uint8 arrays.
    +1 → bit 1, -1 → bit 0
    32x smaller than float32 storage.
    """
    # flatten to 1D and convert to bits
    flat = float_weights.flatten()
    bits = (flat > 0).astype(np.uint8)  # note: +1=1, -1=0

    # pad to multiple of 8 for byte packing
    pad = (8 - len(bits) % 8) % 8
    if pad > 0:
        bits = np.append(bits, np.zeros(pad, dtype=np.uint8))

    # pack 8 bits into each byte
    packed = np.packbits(bits, bitorder='little')
    return packed, float_weights.shape, pad

def unpack_binary_weights(packed, original_shape, pad):
    """
    Unpack bit-packed weights back to ±1 float32.
    Used for verification.
    """
    bits = np.unpackbits(packed, bitorder='little')
    if pad > 0:
        bits = bits[:-pad]
    weights = np.where(bits == 1, 1.0, -1.0).astype(np.float32)
    return weights.reshape(original_shape)

# extract and pack weights from all binary layers
print("Packing binary weights...")
packed_data = {}

for layer_name in ['conv2', 'conv3', 'dense1']:
    layer = binary_model.get_layer(layer_name)
    weights = layer.get_weights()
    kernel  = weights[0]

    # apply final sign() to ensure exactly ±1
    kernel = np.sign(kernel)
    kernel[kernel == 0] = 1.0

    # pack the weights
    packed, shape, pad = pack_binary_weights(kernel)

    packed_data[layer_name] = {
        'kernel_packed': packed,
        'kernel_shape':  shape,
        'kernel_pad':    pad,
        'bias':          weights[1] if len(weights) > 1 else None
    }

    original_kb = kernel.nbytes / 1024
    packed_kb   = packed.nbytes / 1024
    print(f"  {layer_name}: {original_kb:.1f} KB -> {packed_kb:.1f} KB ({original_kb/packed_kb:.0f}x)")

# also extract full precision layers
full_precision_data = {}
for layer_name in ['conv1', 'dense2']:
    try:
        layer = binary_model.get_layer(layer_name)
        full_precision_data[layer_name] = layer.get_weights()
        print(f"  {layer_name}: kept at float32")
    except:
        pass

# extract batchnorm layers
bn_data = {}
for bn_name in ['bn1', 'bn2', 'bn3']:
    try:
        layer = binary_model.get_layer(bn_name)
        bn_data[bn_name] = layer.get_weights()
        print(f"  {bn_name}: batchnorm weights saved")
    except:
        pass

print("\nAll weights extracted and packed.")

In [ ]:
# Next, save as compressed .npz file

save_path = f'{PROJECT_DIR}/models/binary_packed.npz'

# build save dict
save_dict = {}

# binary layers -- packed
for layer_name, data in packed_data.items():
    save_dict[f'{layer_name}_kernel_packed'] = data['kernel_packed']
    save_dict[f'{layer_name}_kernel_shape']  = np.array(data['kernel_shape'])
    save_dict[f'{layer_name}_kernel_pad']    = np.array([data['kernel_pad']])
    if data['bias'] is not None:
        save_dict[f'{layer_name}_bias'] = data['bias']

# full precision layers
for layer_name, weights in full_precision_data.items():
    for j, w in enumerate(weights):
        save_dict[f'{layer_name}_w{j}'] = w

# batchnorm layers
for layer_name, weights in bn_data.items():
    for j, w in enumerate(weights):
        save_dict[f'{layer_name}_w{j}'] = w

np.savez_compressed(save_path, **save_dict)

# check sizes
original_tflite_kb = os.path.getsize(
    f'{PROJECT_DIR}/models/binary_model.tflite'
) / 1024
packed_kb = os.path.getsize(save_path) / 1024

print(f"Size comparison:")
print(f"  FP32 baseline.tflite:     4,252.0 KB")
print(f"  binary_model.tflite:      {original_tflite_kb:.1f} KB (float32 storage)")
print(f"  binary_packed.npz:        {packed_kb:.1f} KB (bit-packed)")
print(f"  Compression vs FP32:      {4252/packed_kb:.1f}x")
print(f"  Compression vs old binary:{original_tflite_kb/packed_kb:.1f}x")
print(f"\nSaved to: {save_path}")

In [ ]:
#  Step 3: Verify unpacking gives back the same weights

print("Verifying bit-packing is lossless...")
all_correct = True

for layer_name, data in packed_data.items():
    # get original weights
    layer    = binary_model.get_layer(layer_name)
    original = np.sign(layer.get_weights()[0])
    original[original == 0] = 1.0

    # unpack the packed weights
    unpacked = unpack_binary_weights(
        data['kernel_packed'],
        data['kernel_shape'],
        data['kernel_pad']
    )

    # check they match exactly
    match = np.all(original == unpacked)
    all_correct = all_correct and match
    print(f"  {layer_name}: {'exact match' if match else 'MISMATCH'}")

print(f"\nAll weights verified: {'PASS' if all_correct else 'FAIL'}")

In [ ]:
# reload test data - just need x_test and y_test
(_, _), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_test  = x_test.astype('float32') / 255.0
y_test  = y_test.flatten()

print(f"x_test shape: {x_test.shape}")
print(f"y_test shape: {y_test.shape}")

class BitPackedBinaryModel:

    def __init__(self, npz_path):
        data = np.load(npz_path)

        # unpack binary layer kernels
        self.kernels = {}
        for layer_name in ['conv2', 'conv3', 'dense1']:
            packed = data[f'{layer_name}_kernel_packed']
            shape  = tuple(data[f'{layer_name}_kernel_shape'])
            pad    = int(data[f'{layer_name}_kernel_pad'][0])
            self.kernels[layer_name] = self._unpack(packed, shape, pad)

        # binary layer biases
        self.biases = {}
        for layer_name in ['conv2', 'conv3', 'dense1']:
            if f'{layer_name}_bias' in data:
                self.biases[layer_name] = data[f'{layer_name}_bias']
            else:
                self.biases[layer_name] = None

        # conv1 (full precision)
        # w0=kernel, w1=bias
        self.conv1_kernel = data['conv1_w0']
        self.conv1_bias   = data['conv1_w1']

        # output dense (full precision)
        # w0=kernel, w1=bias
        self.dense2_kernel = data['dense2_w0']
        self.dense2_bias   = data['dense2_w1']

        # batchnorm weights
        # Keras order: gamma(w0), beta(w1), moving_mean(w2), moving_var(w3)
        self.bn = {}
        for bn_name in ['bn1', 'bn2', 'bn3']:
            self.bn[bn_name] = {
                'gamma':       data[f'{bn_name}_w0'],
                'beta':        data[f'{bn_name}_w1'],
                'moving_mean': data[f'{bn_name}_w2'],
                'moving_var':  data[f'{bn_name}_w3'],
            }

        print(f"Loaded: {npz_path}")
        print(f"Kernel shapes: conv2={self.kernels['conv2'].shape}, "
              f"conv3={self.kernels['conv3'].shape}, "
              f"dense1={self.kernels['dense1'].shape}")

    def _unpack(self, packed, shape, pad):
        bits = np.unpackbits(packed, bitorder='little')
        if pad > 0:
            bits = bits[:-pad]
        weights = np.where(bits == 1, 1.0, -1.0).astype(np.float32)
        return weights.reshape(shape)

    def _batchnorm(self, x, bn_name, eps=1e-3):
        """Apply batchnorm using stored moving statistics (inference mode)."""
        bn = self.bn[bn_name]
        x_norm = (x - bn['moving_mean']) / np.sqrt(bn['moving_var'] + eps)
        return bn['gamma'] * x_norm + bn['beta']

    def _conv2d(self, x, kernel, bias=None):
        """Standard float32 convolution."""
        out = tf.nn.conv2d(
            tf.constant(x), tf.constant(kernel),
            strides=[1,1,1,1], padding='SAME'
        ).numpy()
        if bias is not None:
            out += bias
        return out

    def _binary_conv2d(self, x, kernel, bias=None):
        """Binary convolution - +-1 weights so multiply is exact."""
        out = tf.nn.conv2d(
            tf.constant(x), tf.constant(kernel),
            strides=[1,1,1,1], padding='SAME'
        ).numpy()
        if bias is not None:
            out += bias
        return out

    def _maxpool(self, x):
        return tf.nn.max_pool2d(
            tf.constant(x), ksize=2, strides=2, padding='VALID'
        ).numpy()

    def predict(self, image):
        x = np.expand_dims(image, 0).astype(np.float32)

        # Block 1 - full precision
        x = self._conv2d(x, self.conv1_kernel, self.conv1_bias)
        x = self._batchnorm(x, 'bn1')
        x = np.maximum(0, x)   # relu
        x = self._maxpool(x)

        # Block 2 - binary
        x = self._binary_conv2d(x, self.kernels['conv2'], self.biases['conv2'])
        x = self._batchnorm(x, 'bn2')
        x = np.maximum(0, x)   # relu
        x = self._maxpool(x)

        # Block 3 - binary
        x = self._binary_conv2d(x, self.kernels['conv3'], self.biases['conv3'])
        x = self._batchnorm(x, 'bn3')
        x = np.maximum(0, x)   # relu
        x = self._maxpool(x)

        # flatten
        x = x.reshape(1, -1)

        # dense1 - binary
        x = x @ self.kernels['dense1']
        if self.biases['dense1'] is not None:
            x += self.biases['dense1']
        x = np.maximum(0, x)   # relu

        # output - full precision
        x = x @ self.dense2_kernel + self.dense2_bias

        # softmax
        e = np.exp(x[0] - np.max(x[0]))
        probs = e / e.sum()

        return int(np.argmax(probs)), probs


# reload and test
packed_model = BitPackedBinaryModel(
    f'{PROJECT_DIR}/models/binary_packed.npz'
)

# sanity check on first 10 images
CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']

print("\nSanity check on first 10 images:")
print(f"{'#':<5} {'True':<15} {'Predicted':<15} {'Conf':>8} {'OK?'}")
print("-" * 50)
for i in range(10):
    pred, probs = packed_model.predict(x_test[i])
    true_label  = CLASSES[y_test[i]]
    pred_label  = CLASSES[pred]
    correct     = 'yes' if pred == y_test[i] else 'no'
    print(f"{i:<5} {true_label:<15} {pred_label:<15} {probs[pred]*100:>7.1f}%  {correct}")

In [ ]:
# Full accuracy evaluation on all 10,000 test images
print("Evaluating on full test set (10,000 images)...")
print("This will take a few minutes...\n")

correct   = 0
latencies = []

for i in range(len(x_test)):
    start = time.perf_counter()
    pred, _ = packed_model.predict(x_test[i])
    latencies.append((time.perf_counter() - start) * 1000)

    if pred == y_test[i]:
        correct += 1

    if (i + 1) % 1000 == 0:
        print(f"  {i+1}/10000 - accuracy so far: {correct/(i+1)*100:.2f}%")

accuracy     = correct / len(x_test)
avg_delay_ms = np.mean(latencies)
p95_delay_ms = np.percentile(latencies, 95)
packed_kb    = os.path.getsize(
    f'{PROJECT_DIR}/models/binary_packed.npz'
) / 1024

print(f"\n{'='*55}")
print(f"BIT-PACKED BINARY MODEL RESULTS")
print(f"{'='*55}")
print(f"  Accuracy:              {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  Avg time delay:        {avg_delay_ms:.3f} ms (Colab T4)")
print(f"  P95 time delay:        {p95_delay_ms:.3f} ms (Colab T4)")
print(f"  Model size:            {packed_kb:.1f} KB")
print(f"  Compression vs FP32:   {4252/packed_kb:.1f}x")
print(f"{'='*55}")
print()
print("Comparison with other variants:")
print(f"  FP32:            84.63% | 4,252.0 KB | baseline")
print(f"  PTQ:             83.27% |   365.6 KB | 11.6x")
print(f"  QAT:             83.73% |   365.6 KB | 11.6x")
print(f"  Binary (old):    80.93% | 1,401.9 KB |  3.0x")
print(f"  Binary (packed): {accuracy*100:.2f}% | {packed_kb:7.1f} KB | {4252/packed_kb:.1f}x")

# save results
import csv
results_path = f'{PROJECT_DIR}/results/clean_eval.csv'
file_exists  = os.path.isfile(results_path)

with open(results_path, 'a', newline='') as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(['variant','accuracy','avg_latency_ms',
                        'p95_latency_ms','size_kb'])
    writer.writerow([
        'Binary_Packed',
        round(accuracy, 4),
        round(avg_delay_ms, 3),
        round(p95_delay_ms, 3),
        round(packed_kb, 2)
    ])

print(f"\nSaved to {results_path}")

In [ ]:
pip install ai-edge-litert

In [ ]:
# Test: compare packed model vs original tflite because can't deploy the bit packed on Pi
# run both on the same 100 images and see where they disagree

from ai_edge_litert.interpreter import Interpreter

# load original binary tflite for comparison
orig_interp = Interpreter(
    model_path=f'{PROJECT_DIR}/models/binary_model.tflite'
)
orig_interp.allocate_tensors()
orig_inp = orig_interp.get_input_details()[0]
orig_out = orig_interp.get_output_details()[0]

print("Comparing packed model vs original tflite on 100 images:")
print(f"{'#':<5} {'True':<12} {'Original':<12} {'Packed':<12} {'Match?'}")
print("-" * 55)

orig_correct   = 0
packed_correct = 0
agree          = 0

for i in range(100):
    image = x_test[i:i+1].astype(np.float32)

    # original tflite prediction
    orig_interp.set_tensor(orig_inp['index'], image)
    orig_interp.invoke()
    orig_pred = int(np.argmax(orig_interp.get_tensor(orig_out['index'])[0]))

    # packed model prediction
    packed_pred, _ = packed_model.predict(x_test[i])

    true_label = y_test[i]
    match = 'yes' if orig_pred == packed_pred else 'no'

    if orig_pred == true_label:
        orig_correct += 1
    if packed_pred == true_label:
        packed_correct += 1
    if orig_pred == packed_pred:
        agree += 1

    if orig_pred != packed_pred:
        print(f"{i:<5} {CLASSES[true_label]:<12} "
              f"{CLASSES[orig_pred]:<12} {CLASSES[packed_pred]:<12} {match}")

print(f"\nOut of 100 images:")
print(f"  Original tflite accuracy: {orig_correct}/100")
print(f"  Packed model accuracy:    {packed_correct}/100")
print(f"  Agreement between models: {agree}/100")